# Maximum Likelihood Estimation

In our discussion of the [data-generating process](https://github.com/aecins/tutorials/blob/main/ml/data_generating_process.ipynb), we assumed that our data is drawn i.i.d. from an unknown true distribution $p^*(X, Y)$, and that all we ever get to see is a finite dataset $\mathcal{D} = \{(x_1, y_1), \dots, (x_N, y_N)\}$. Learning from data means using $\mathcal{D}$ to find a model that captures what the data-generating process does. In this discussion we make this idea precise and arrive at *maximum likelihood estimation*, the principle behind the training of most machine learning models.

More precisely, we are looking for a **parametric probabilistic model** $p_\theta$ whose distribution approximates the distribution of the true data-generating process as closely as possible. "As closely as possible", because the true data-generating process is almost always more complex than any model we write down, so no model will match it exactly (this is called *model misspecification*).

We proceed in steps. First, we explain what a parametric probabilistic model is. Next, we see why we cannot directly measure how close a model is to the true distribution, and why we instead need a score that can be computed from the dataset alone. That score is the *likelihood*: we build an intuition for what it measures and use it to define maximum likelihood estimation. Finally, we show why maximum likelihood estimation works: maximizing the likelihood brings the model's distribution close to the true distribution, as measured by the *Kullback–Leibler (KL) divergence*.

## Parametric Probabilistic Models

The models we consider have two defining properties: they are **probabilistic** and they are **parametric**.

### Probabilistic Models

A probabilistic model defines a probability distribution that we can evaluate at any data point: for a given sample, we can compute the probability the model assigns to it.

**Which distribution the model represents depends on the task.** A *discriminative* model, such as a classifier, represents the conditional distribution $p_\theta(Y \mid X)$. A *generative* model represents the joint distribution $p_\theta(X, Y)$, usually as $p_\theta(X \mid Y)\, p_\theta(Y)$. An *unsupervised* model, such as an image generator or a language model, represents only the marginal distribution $p_\theta(X)$.

Note that a probabilistic model does not have to *output* probabilities directly. A neural network by itself outputs numbers, and it becomes a probabilistic model through how we *interpret* those numbers:

* **Classification:** A final *softmax* layer turns the network's outputs into non-negative numbers that sum to 1, one per species. We interpret them as the class probabilities $p_\theta(Y = k \mid X = x)$.
* **Regression:** The network outputs a single number $f_\theta(x)$, which we interpret as the mean of a distribution over targets, for example a Gaussian $p_\theta(Y \mid X = x) = \mathcal{N}\big(f_\theta(x), \sigma^2\big)$. The network never outputs a probability, but we can still evaluate $p_\theta(y \mid x)$ for any target $y$.

Conversely, not every model qualifies. A classic **support vector machine (SVM)** outputs a decision ("Dog" or "Cat") without defining any probability distribution. A **generative adversarial network (GAN)** does define a distribution over images (we can draw new images from it), but it provides no way to evaluate the probability of a given image. The tools developed in this discussion apply only to models whose probabilities we can evaluate.

### Parametric Models

A **parametric model** is a family of distributions whose members are described by a fixed set of numbers, the **parameters** $\theta$. The form of the model, for example a neural network architecture, is chosen in advance and determines which distributions can be represented at all. The values of the parameters, for example the network's weights, then pick out one specific distribution $p_\theta$ from this family. For a classifier, changing the weights changes the probabilities that the classifier assigns to each class for a given feature vector.

In everyday language, the word "model" is used for both: sometimes it means the family (the architecture), sometimes a single member of it (the architecture with specific weights). In this discussion, we use *model family* for the former and *model* $p_\theta$ for the latter.

**Learning** then means searching for the parameters $\theta$ whose distribution is as close as possible to the true distribution, $p_\theta(X, Y) \approx p^*(X, Y)$, or to the part of it that the model represents, such as $p_\theta(Y \mid X) \approx p^*(Y \mid X)$ for a classifier.

There are also *non-parametric* methods, such as k-nearest neighbors or kernel density estimation, which do not fix the form of the distribution in advance and instead grow with the data. They are outside the scope of this discussion.

## The Problem: Scoring a Model Without Knowing $p^*$

Given a model family, a specific choice of parameters $\theta$ and a dataset $\mathcal{D}$, we want to know: **how well does $p_\theta$ approximate the true data-generating process?**

Ideally, we would measure the distance between the model's distribution $p_\theta$ and the true distribution $p^*$ directly. Such measures of distance between distributions exist, but they all require us to know $p^*$, and $p^*$ is exactly what we do not know. All we have is the dataset $\mathcal{D}$: a finite collection of samples drawn from $p^*$.

So we need a way to **score a candidate $\theta$ using only the data**. The score that does this is the **likelihood**. We first build an intuition for what it measures. Later, we will show that it is indeed connected to a proper measure of distance between $p_\theta$ and $p^*$: the KL divergence.

## The Likelihood Function

The idea behind the likelihood is simple: **a good model should find the data we actually observed probable.** Formally, the **likelihood** of the parameters $\theta$ is the probability that the model $p_\theta$ assigns to the observed dataset $\mathcal{D}$, viewed as a function of $\theta$:

$$L(\theta) = p_\theta(\mathcal{D})$$

Because we assume the samples are drawn **i.i.d.**, we can factorize the probability of the whole dataset into a product of per-sample probabilities:

$$L(\theta) = p_\theta(\mathcal{D}) = \prod_{i=1}^N p_\theta(x_i, y_i)$$

To get an intuition for what this number means, imagine that the model $p_\theta$ *were* the true data-generating process, and that we repeated our data collection with it. The likelihood is the probability that this repetition would produce exactly the dataset we actually observed. A model with a higher likelihood is one that would more often reproduce what we saw.

Two consequences are important to keep in mind:

* **Only comparisons are meaningful.** Each additional sample is one more condition that has to hold at the same time, so the likelihood shrinks rapidly as the dataset grows: under any model, any specific large dataset is improbable. Its absolute value therefore says little on its own. What carries information is how $L(\theta)$ compares between different values of $\theta$ on the same dataset.
* **The likelihood is not a probability distribution over $\theta$.** $L(\theta)$ is the probability of the *data* given the parameters, not the probability that $\theta$ is the correct parameter. Summed or integrated over all values of $\theta$, it does not, in general, give 1.

## Maximum Likelihood Estimation

The likelihood gives us a score for every candidate $\theta$. **Maximum likelihood estimation (MLE)** chooses the parameters with the highest score, that is, the parameters under which the observed data is most probable:

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta L(\theta) = \arg\max_\theta \prod_{i=1}^N p_\theta(x_i, y_i)$$

Training a model with MLE is therefore a search over the model family: among all distributions that the family can represent, find the one that makes the observed dataset most probable.

### Maximizing the Likelihood with Gradients

To find the maximum, we need the **gradient** of the likelihood with respect to the parameters, $\nabla_\theta L(\theta)$. It tells us how the likelihood changes when we change the parameters slightly. For simple models, we can set the gradient to zero and solve for $\theta$ in closed form. For neural networks this is not possible. Instead, we use *gradient-based optimization*: starting from some initial value of $\theta$, we repeatedly compute the gradient and take a small step in its direction,

$$\theta \leftarrow \theta + \eta \, \nabla_\theta L(\theta)$$

where the step size $\eta$ is called the *learning rate*.

To compute the gradient, let's write $p_i(\theta) = p_\theta(x_i, y_i)$ for the probability that the model assigns to the $i$-th sample. The likelihood is the product $L(\theta) = p_1(\theta)\, p_2(\theta) \cdots p_N(\theta)$. Using the product rule, we get:

$$\nabla_\theta L(\theta) = \sum_{i=1}^N \Big( \nabla_\theta p_i(\theta) \prod_{j \neq i} p_j(\theta) \Big)$$

This form can be simplified by multiplying each term in the sum by $\frac{p_i(\theta)}{p_i(\theta)} = 1$:

$$
\begin{aligned}
\nabla_\theta L(\theta) &= \sum_{i=1}^N \Big( \nabla_\theta p_i(\theta) \cdot \frac{p_i(\theta)}{p_i(\theta)} \prod_{j \neq i} p_j(\theta) \Big) \\
&= \sum_{i=1}^N \Big( \frac{\nabla_\theta p_i(\theta)}{p_i(\theta)} \cdot \Big[ p_i(\theta) \prod_{j \neq i} p_j(\theta) \Big] \Big) \\
&= \sum_{i=1}^N \Big( \frac{\nabla_\theta p_i(\theta)}{p_i(\theta)} \cdot \prod_{j=1}^N p_j(\theta) \Big) \\
&= \sum_{i=1}^N \Big( \frac{\nabla_\theta p_i(\theta)}{p_i(\theta)} \cdot L(\theta) \Big) \\
&= L(\theta) \sum_{i=1}^N \frac{\nabla_\theta p_i(\theta)}{p_i(\theta)}
\end{aligned}
$$

### Why the Gradient of the Likelihood Is Problematic

Computing the gradient of the likelihood has two main problems:

* **Numerical stability.** As the last line of the derivation shows, the entire gradient is multiplied by $L(\theta)$, a product of $N$ probabilities. For classification, these are values between 0 and 1. For a large dataset, this product becomes vanishingly small and cannot be accurately represented by floating-point numbers. Its value collapses to zero, which leads to zero gradients and no learning updates.
* **Decomposability.** To calculate the gradient contribution of any one sample, we need the $L(\theta)$ term, which involves the whole dataset. With millions of samples, every single optimization step would require a full pass over the data, so computing the direction of each step takes a very long time. This is a serious issue for gradient-based optimization, where the objective has a complicated shape and has to be climbed in many small steps to reach the maximum. We also cannot compute the gradient from a small random subset of the data (a *mini-batch*) instead: it would be scaled by the likelihood of the mini-batch rather than of the whole dataset. These differ by many orders of magnitude (for example, $0.9^{100} \approx 10^{-5}$ versus $0.9^{10{,}000} \approx 10^{-458}$), so the mini-batch gradient is not an estimate of the full gradient.

## The Log-Likelihood

These problems can be avoided by using the **log-likelihood** instead:

$$\ell(\theta) = \log L(\theta) = \sum_{i=1}^N \log p_\theta(x_i, y_i)$$

The logarithm turns the product into a sum, and the derivative of a sum is the sum of the derivatives. The gradient of the log-likelihood is therefore:

$$\nabla_\theta \ell(\theta) = \sum_{i=1}^N \nabla_\theta \log p_i(\theta) = \sum_{i=1}^N \frac{\nabla_\theta p_i(\theta)}{p_i(\theta)}$$

This is the same sum as in the gradient of the likelihood, but without the factor $L(\theta)$ in front.

### How the Log-Likelihood Solves These Problems

* **Numerical stability.** The log-likelihood and its gradient are sums of $N$ per-sample terms. For classification, where the probabilities lie between 0 and 1, each term $\log p_i(\theta)$ lies in $(-\infty, 0]$ and is a moderate negative number unless the model assigns the sample a probability very close to 0. A sum of such terms is a large negative number that floating-point arithmetic represents without problems, so the underflow issue disappears.

* **Decomposability.** Each term of the sum depends on a single sample only. We can therefore estimate the full gradient from a small random subset $\mathcal{B}$ of the dataset, a *mini-batch*:

  $$\nabla_\theta \ell(\theta) \approx \frac{N}{|\mathcal{B}|} \sum_{i \in \mathcal{B}} \nabla_\theta \log p_\theta(x_i, y_i)$$

  On average over the random choice of the mini-batch, this estimate equals the full gradient, so each optimization step only needs a few samples instead of the entire dataset. This is the idea behind *stochastic gradient descent*, the workhorse of training neural networks.

### Maximizing the Log-Likelihood Maximizes the Likelihood

Switching to the log-likelihood does not change the solution:

* **Same maximum.** The logarithm is strictly increasing, so if $L(\theta_1) > L(\theta_2)$ then also $\ell(\theta_1) > \ell(\theta_2)$. The parameters that maximize $L(\theta)$ are therefore exactly the parameters that maximize $\ell(\theta)$:

  $$\hat{\theta}_{\text{MLE}} = \arg\max_\theta L(\theta) = \arg\max_\theta \ell(\theta)$$

* **Same gradient direction.** By the chain rule, $\nabla_\theta \ell(\theta) = \dfrac{\nabla_\theta L(\theta)}{L(\theta)}$. Since $L(\theta) > 0$, the gradient of the log-likelihood points in the same direction as the gradient of the likelihood, and it is zero exactly where the gradient of the likelihood is zero. The log only rescales the gradient by $1 / L(\theta)$, which removes precisely the factor that caused both problems.

### Average and Negative Log-Likelihood

In practice, the log-likelihood is used in a slightly modified form. Dividing by $N$ makes the value comparable across datasets of different sizes, and flipping the sign turns maximization into minimization, which is what optimization libraries expect. Neither change affects the optimal parameters. The result is the **negative log-likelihood (NLL)**:

$$\text{NLL}(\theta) = -\frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i, y_i), \qquad \hat{\theta}_{\text{MLE}} = \arg\min_\theta \text{NLL}(\theta)$$

Training a model by minimizing the NLL as its loss function *is* maximum likelihood estimation.

## Why MLE Works: The KL Divergence

At the start, we set out to find the model whose distribution is as close as possible to the true distribution $p^*$, but we could not measure this closeness directly because $p^*$ is unknown. We now show that minimizing the NLL does this, at least approximately: it brings $p_\theta$ close to $p^*$, as measured by the Kullback–Leibler divergence.

### The KL Divergence

The **Kullback–Leibler (KL) divergence** measures how different a distribution $q$ is from a distribution $p$. Writing $Z$ for whatever the distributions describe (for us, the pair $(X, Y)$), it is defined as:

$$\mathrm{KL}(p \,\|\, q) = \mathbb{E}_{Z \sim p}\left[\log \frac{p(Z)}{q(Z)}\right]$$

For discrete $Z$ the expectation is a sum, $\sum_z p(z) \log \frac{p(z)}{q(z)}$, and for continuous $Z$ it is an integral. Intuitively, the KL divergence draws samples from $p$ and asks, on average, how much more probable $p$ finds them than $q$ does. It has three important properties:

* $\mathrm{KL}(p \,\|\, q) \geq 0$ for all distributions $p$ and $q$.
* $\mathrm{KL}(p \,\|\, q) = 0$ if and only if $p = q$.
* It is **not symmetric**: in general, $\mathrm{KL}(p \,\|\, q) \neq \mathrm{KL}(q \,\|\, p)$. It is therefore not a distance in the strict mathematical sense, but it behaves like one for our purposes: it is zero exactly when the two distributions match, and it grows as they become more different.

We use $\mathrm{KL}(p^* \,\|\, p_\theta)$, with the true distribution first: the average is taken over data drawn from $p^*$, and the model is penalized wherever it assigns low probability to data that the true process actually produces.

### From the KL Divergence to the Likelihood

Splitting the logarithm of the ratio into a difference gives:

$$\mathrm{KL}(p^* \,\|\, p_\theta) = \mathbb{E}_{p^*}\big[\log p^*(X, Y)\big] - \mathbb{E}_{p^*}\big[\log p_\theta(X, Y)\big]$$

The first term depends only on the true distribution, not on $\theta$. Minimizing the KL divergence over $\theta$ is therefore the same as maximizing the second term, the expected log-probability that the model assigns to data from the true distribution:

$$\arg\min_\theta \mathrm{KL}(p^* \,\|\, p_\theta) = \arg\max_\theta \mathbb{E}_{p^*}\big[\log p_\theta(X, Y)\big]$$

We still cannot compute this expectation, because it is taken over the unknown $p^*$. But we have samples from $p^*$: our dataset. By the Law of Large Numbers, the average over the dataset converges to the expectation as the dataset grows:

$$-\text{NLL}(\theta) = \frac{1}{N} \sum_{i=1}^N \log p_\theta(x_i, y_i) \xrightarrow{N \to \infty} \mathbb{E}_{p^*}\big[\log p_\theta(X, Y)\big]$$

Putting the pieces together: **minimizing the NLL on the dataset is a data-based approximation of minimizing the KL divergence to the true distribution.** This resolves the problem we started with. The KL divergence is a proper measure of how close $p_\theta$ is to $p^*$, but it requires $p^*$. Maximum likelihood estimation replaces the expectation under $p^*$ with an average over the samples we have.

### Caveats: Finite Data and Overfitting

The connection to $p^*$ relies on the Law of Large Numbers, which only holds as $N \to \infty$. With a finite dataset, the picture is more subtle:

* **MLE fits the empirical distribution.** Replacing $p^*$ by the empirical distribution $\hat{p}_{\text{data}}$ in the derivation above shows that, for discrete data and up to a term that does not depend on $\theta$, the NLL is exactly $\mathrm{KL}(\hat{p}_{\text{data}} \,\|\, p_\theta)$. On a finite dataset, MLE therefore brings the model close to the *dataset*, not directly to $p^*$.
* **An unrestricted model would memorize the data.** If the model could represent any distribution, the best choice would be $\hat{p}_{\text{data}}$ itself: with all samples distinct, probability $\frac{1}{N}$ on each training sample and 0 everywhere else. Such a model assigns probability 0 to every new data point and has learned nothing about $p^*$. It is the restriction to a parametric family that prevents this: a network with a limited number of weights has to spread probability smoothly, so that similar data points receive similar probabilities. If the family is too flexible, the model moves toward memorizing the training data, which is called **overfitting**.
* **Training performance is optimistic.** Because the parameters are chosen to maximize the likelihood on $\mathcal{D}$, they look better on $\mathcal{D}$ than they really are on new data from $p^*$. This is why a trained model is evaluated on separate, *held-out* data that was not used for training.

## Summary

* We look for a **parametric probabilistic model** $p_\theta$ whose distribution is as close as possible to the true distribution $p^*$.
* Since $p^*$ is unknown, we score candidate parameters with the **likelihood** $L(\theta) = \prod_i p_\theta(x_i, y_i)$, the probability of the observed data under the model. **Maximum likelihood estimation** chooses the parameters with the highest likelihood.
* In practice, we minimize the **negative log-likelihood**, which has the same optimum but is numerically stable and decomposes into per-sample terms, so that it can be optimized with mini-batch gradient descent.
* Minimizing the NLL approximates minimizing the **KL divergence** $\mathrm{KL}(p^* \,\|\, p_\theta)$ to the true distribution. This is why maximum likelihood estimation works. As we will show later, for a classifier the NLL is the familiar **cross-entropy loss**.
* On a finite dataset, MLE fits the empirical distribution. A suitably restricted model family and evaluation on held-out data protect against **overfitting**.